In [1]:
import os
import dotenv 

dotenv.load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage


model = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

In [5]:
model.invoke([HumanMessage(content="Hi, My name is Anand Mansabdar and I am currently learning GenAI")])

AIMessage(content='Nice to meet you, Anand Mansabdar.  Congratulations on taking the first step into learning GenAI (General Artificial Intelligence). This is an exciting and rapidly evolving field that has the potential to revolutionize numerous industries and aspects of our lives. \n\nWhat specific aspects of GenAI are you interested in learning about? Are you focusing on the theoretical foundations, the applications, or perhaps the implementation of GenAI systems using programming languages like Python or other tools?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 52, 'total_tokens': 145, 'completion_time': 0.171693186, 'completion_tokens_details': None, 'prompt_time': 0.002465342, 'prompt_tokens_details': None, 'queue_time': 0.157652217, 'total_time': 0.174158528}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider':

In [6]:
model.invoke(
  [
    HumanMessage(content="Hi, My name is Anand Mansabdar and I am currently learning GenAI"),
    AIMessage(content='Nice to meet you, Anand Mansabdar.  Congratulations on taking the first step into learning GenAI (General Artificial Intelligence). This is an exciting and rapidly evolving field that has the potential to revolutionize numerous industries and aspects of our lives. \n\nWhat specific aspects of GenAI are you interested in learning about? Are you focusing on the theoretical foundations, the applications, or perhaps the implementation of GenAI systems using programming languages like Python or other tools?'),
    HumanMessage(content="Can you tell me my name and what am I currently learning?")
  ]
)

AIMessage(content='Your name is Anand Mansabdar, and you are currently learning GenAI (General Artificial Intelligence).', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 167, 'total_tokens': 189, 'completion_time': 0.021269129, 'completion_tokens_details': None, 'prompt_time': 0.011689474, 'prompt_tokens_details': None, 'queue_time': 0.047976845, 'total_time': 0.032958603}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea648-1338-7903-8212-f1bdda10851e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 167, 'output_tokens': 22, 'total_tokens': 189})

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

In [10]:
store = {} # To store the session ids

def get_session_history(session_id:str) -> BaseChatMessageHistory:
  if session_id not in store:
    store[session_id] = ChatMessageHistory()
  return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [9]:
config = {"configurable": {"session_id": "chat1"}}

In [12]:
response = with_message_history.invoke(
  [
    HumanMessage(content="hi, My name is Anand Mansabdar and I am a B.Tech UG")
  ],config=config
)

In [13]:
response.content

'Nice to meet you, Anand Mansabdar. It seems like you introduced yourself earlier. What would you like to talk about or ask? Do you have any questions regarding B.Tech, college life, or engineering in general?'

In [14]:
with_message_history.invoke([
  HumanMessage(content="What is my name?")
], config=config)

AIMessage(content='Your name is Anand Mansabdar.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 186, 'total_tokens': 196, 'completion_time': 0.009462672, 'completion_tokens_details': None, 'prompt_time': 0.015023986, 'prompt_tokens_details': None, 'queue_time': 0.048305173, 'total_time': 0.024486658}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea660-66a5-70f2-b7e6-385638c28500-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 186, 'output_tokens': 10, 'total_tokens': 196})

In [15]:
# Changing the config ie., changing the session id
new_config = {"configurable": {"session_id": "chat2"}}
with_message_history.invoke([
  HumanMessage(content="What is my name?")
], config=new_config)

AIMessage(content="I don't have any information about your name. I'm a conversational AI, and our conversation has just started. I don't retain any information about previous conversations or users. If you'd like to share your name with me, I'd be happy to use it in our conversation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 40, 'total_tokens': 99, 'completion_time': 0.07576542, 'completion_tokens_details': None, 'prompt_time': 0.009889204, 'prompt_tokens_details': None, 'queue_time': 0.418160842, 'total_time': 0.085654624}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea661-3c15-71d2-acf6-051c0d1df510-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 59, 'total_tokens': 99})

In [18]:
prompt = ChatPromptTemplate.from_messages([
  ("system", "You are a helpful assistant. Answer all the questions with highest possible accuracy."),
  MessagesPlaceholder(variable_name="messages")
])

In [19]:
chain = prompt | model | StrOutputParser()

chain.invoke({"messages": [HumanMessage(content="Hi. My name is Anand Mansabdar")]})

"Nice to meet you, Anand Mansabdar. I'm here to assist you with any questions or information you may need. How can I help you today?"

In [20]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [21]:
config = {"configurable": {"session_id": "chat3"}}

response = with_message_history.invoke(
  [HumanMessage(content="Hi. My name is Anand Mansabdar")],
  config=config
)

response

"Nice to meet you, Anand Mansabdar. I'm happy to assist you with any questions or information you may need. Is there something specific on your mind, or would you like to start with a general conversation?"